In [1]:
!pip install crewai pandas numpy requests ollama python-dotenv
import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
from typing import Dict, List, Any
import ollama
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
import warnings
warnings.filterwarnings('ignore')

print("All dependencies installed successfully!")
try:
    response = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': 'Hello, are you working?'}])
    print("Ollama connection successful!")
    print(f"Response: {response['message']['content'][:50]}...")
except Exception as e:
    print(f"Ollama connection failed: {e}")
    print("Please ensure Ollama is running with: ollama serve")

All dependencies installed successfully!
Ollama connection successful!
Response: Yes, I'm here and ready to help. How can I assist ...


In [4]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

class ExcelFrappeDB:
    """Load business context from a single-sheet Excel file and organize by element type."""
    def __init__(self, excel_path: str):
        self.excel_path = excel_path
        self.load_data_from_excel()

    def load_data_from_excel(self):
        """Read and split data from a single sheet based on element types"""
        df = pd.read_excel(self.excel_path)
        df['Element Type'] = df['Element Type'].astype(str).str.lower().str.strip()
        df['Impact filters'] = df['Impact filters'].astype(str).str.replace(' ', '').str.lower()
        df['Product'] = df['Product'].astype(str).str.strip()
        
        self.promotions = df[df['Element Type'] == 'promotion'].to_dict(orient='records')
        self.stockouts = df[df['Element Type'] == 'stockout'].to_dict(orient='records')
        self.distributor_activity = df[df['Element Type'] == 'distributor'].to_dict(orient='records')
        self.events = df[df['Element Type'].isin(['seasonal', 'temperature'])].to_dict(orient='records')
        self.products = list(df['Product'].dropna().unique())
        self.regions = list(df['Impact filters'].dropna().apply(lambda x: self.extract_region(x)).dropna().unique())
        
        print(f"Loaded: {len(self.products)} products, {len(self.regions)} regions, "
              f"{len(self.promotions)} promotions, {len(self.stockouts)} stockouts, "
              f"{len(self.distributor_activity)} distributor activities, {len(self.events)} events")

    def extract_region(self, filter_str):
        """Extract region from 'Impact filters' like 'product=AC-X1,region=North'"""
        try:
            for part in filter_str.split(','):
                if 'region=' in part:
                    return part.split('=')[1].strip()
        except:
            pass
        return None

    def format_date_string(self, date_value):
        """Convert various date formats to 'DD-MMM' format"""
        if pd.isna(date_value) or date_value is None:
            return None
            
        # If it's already a string, try to parse it
        if isinstance(date_value, str):
            date_value = date_value.strip()
            # If it's already in DD-MMM format, return as is
            if len(date_value.split('-')) == 2 and len(date_value.split('-')[0]) <= 2:
                return date_value
        
        # If it's a pandas Timestamp or datetime object
        if isinstance(date_value, (pd.Timestamp, datetime)):
            return date_value.strftime("%d-%b")
        
        # Try to parse string dates
        try:
            # Try various date formats
            for fmt in ["%Y-%m-%d", "%d-%m-%Y", "%m/%d/%Y", "%d/%m/%Y"]:
                try:
                    parsed_date = datetime.strptime(str(date_value), fmt)
                    return parsed_date.strftime("%d-%b")
                except ValueError:
                    continue
        except:
            pass
        
        return str(date_value)

    def get_impact_weeks(self, entry, year=2025):
        """Calculate weeks affected by an event based on impact window type"""
        try:
            impact_type = str(entry.get('Impact Window Type', '')).lower().strip()
            
            if impact_type in ['fixed yearly occurrence', 'from event']:
                start_str = self.format_date_string(entry.get('Impact Window Start Fixed'))
                end_str = self.format_date_string(entry.get('Impact Window End Fixed'))
                
            elif impact_type == 'relative':
                base_str = self.format_date_string(entry.get('Impact Window Start Fixed'))
                start_offset = int(entry.get('Impact Window Start', 0) or 0)
                end_offset = int(entry.get('Impact Window End', 0) or 0)
                
                if not base_str:
                    return []
                    
                try:
                    base_date = datetime.strptime(f"{base_str}-{year}", "%d-%b-%Y")
                    start_date = base_date + timedelta(weeks=start_offset)
                    end_date = base_date + timedelta(weeks=end_offset)
                    start_str = start_date.strftime("%d-%b")
                    end_str = end_date.strftime("%d-%b")
                except ValueError as e:
                    print(f"Error parsing relative date {base_str}: {e}")
                    return []
            else:
                return []
            
            if not start_str or not end_str:
                return []
                
            try:
                start_date = datetime.strptime(f"{start_str}-{year}", "%d-%b-%Y")
                end_date = datetime.strptime(f"{end_str}-{year}", "%d-%b-%Y")
                
                start_week = start_date.isocalendar()[1]
                end_week = end_date.isocalendar()[1]
                
                # Handle year boundary crossing
                if end_week < start_week:
                    end_week += 52  # Add weeks for next year
                
                return list(range(start_week, end_week + 1))
                
            except ValueError as e:
                print(f"Error parsing dates {start_str}-{year} to {end_str}-{year}: {e}")
                return []
                
        except Exception as e:
            print(f"Error in get_impact_weeks for entry: {e}")
            return []

    def match_entry(self, entry, product_code, region, week, year=2025):
        """Check if entry matches product, region, and week"""
        try:
            impact_filters = str(entry.get('Impact filters', ''))
            if not impact_filters or impact_filters == 'nan':
                return False
                
            filters = {}
            for item in impact_filters.split(','):
                if '=' in item:
                    key, value = item.split('=', 1)
                    filters[key.strip()] = value.strip()
            
            # Check product match
            product_match = (
                filters.get('product', '').lower() == product_code.lower() or
                filters.get('item_group', '').lower() == product_code.lower()
            )
            
            # Check region match
            region_match = filters.get('region', '').lower() == region.lower()
            
            # Check week match
            impact_weeks = self.get_impact_weeks(entry, year)
            week_match = week in impact_weeks
            
            return product_match and region_match and week_match
            
        except Exception as e:
            print(f"Error in match_entry: {e}")
            return False

    def get_context(self, product_code: str, region: str, week: int, year: int = 2025) -> dict:
        """Fetch business context for given product, region, and week"""
        product_code = product_code.strip()
        region = region.strip()
        
        try:
            context = {
                'product_code': product_code,
                'region': region,
                'week': week,
                'promotions': [p for p in self.promotions if self.match_entry(p, product_code, region, week, year)],
                'stockouts': [s for s in self.stockouts if self.match_entry(s, product_code, region, week, year)],
                'events': [e for e in self.events if self.match_entry(e, product_code, region, week, year)],
                'distributor_activity': next((d for d in self.distributor_activity if self.match_entry(d, product_code, region, week, year)), {})
            }
            
            print(f"\nContext for product={product_code.upper()}, region={region.title()}, week={week}:")
            print(f"  Promotions: {len(context['promotions'])}, Stockouts: {len(context['stockouts'])}, "
                  f"Events: {len(context['events'])}, Distributor activity: {'Found' if context['distributor_activity'] else 'None'}")
            
            return context
            
        except Exception as e:
            print(f"Error getting context: {e}")
            return {
                'product_code': product_code,
                'region': region,
                'week': week,
                'promotions': [],
                'stockouts': [],
                'events': [],
                'distributor_activity': {}
            }

def pandas_to_serializable(obj):
    """Convert pandas objects to JSON-serializable format"""
    if isinstance(obj, pd.DataFrame):
        obj = obj.where(pd.notnull(obj), None)
        return obj.to_dict(orient='records')
    if isinstance(obj, pd.Series):
        obj = obj.where(pd.notnull(obj), None)
        return obj.to_dict()
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, pd.NaT.__class__):
        return None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: pandas_to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [pandas_to_serializable(i) for i in obj]
    return obj

# Test the fixed code
if __name__ == "__main__":
    excel_path = r"../backend/data/dummy_data.xlsx"
    db = ExcelFrappeDB(excel_path)
    sample = db.get_context('AC-X1', 'North', 35)
    serializable_sample = pandas_to_serializable(sample)
    print(json.dumps(serializable_sample, indent=2))

Loaded: 8 products, 3 regions, 2 promotions, 1 stockouts, 1 distributor activities, 4 events

Context for product=AC-X1, region=North, week=35:
  Promotions: 0, Stockouts: 0, Events: 1, Distributor activity: None
{
  "product_code": "AC-X1",
  "region": "North",
  "week": 35,
  "promotions": [],
  "stockouts": [],
  "events": [
    {
      "Product": "Back to School - AC-X1 North",
      "Element Type": "seasonal",
      "Event Description": "Increased purchases for electronics in Back to School campaign in North",
      "Impact Window Type": "Fixed Yearly Occurrence",
      "Impact Window Start": NaN,
      "Impact Window End": NaN,
      "Impact Window Start Fixed": "2025-08-25T00:00:00",
      "Impact Window End Fixed": "2025-09-05T00:00:00",
      "Response Type": "Categorical",
      "Impact Magnitude Formula": NaN,
      "Impact filters": "product=ac-x1,region=north",
      "Activity Level": "Low"
    }
  ],
  "distributor_activity": {}
}


In [6]:
from crewai_tools import BaseTool

global_mock_db = None

class ContextFetcherTool(BaseTool):
    """Custom tool to fetch business context from mock database"""
    name: str = "Business Context Fetcher"
    description: str = "Fetches business context including promotions, stockouts, distributor activity, and events for a given product, region, and week"

    def _run(self, product_code: str, region: str, week: int) -> str:
        """Fetch and format business context with multipliers"""
        try:
            context = global_mock_db.get_context(product_code, region, week)
            output = f"CONTEXT FOR {product_code} in {region} - Week {week}:\n"

            for key, label in [('promotions', 'PROMOTIONS'), ('stockouts', 'STOCKOUTS'), ('events', 'EVENTS')]:
                output += f"\n{label}:\n"
                items = context[key] if isinstance(context[key], list) else [context[key]] if context[key] else []
                for item in items:
                    desc = item.get('Event Description', 'Unknown')
                    if item.get('Response Type', '').lower() == 'numerical':
                        formula = item.get('Impact Magnitude Formula', '')
                        multiplier = 1.0
                        if 'Multiplier' in formula:
                            multiplier = float(formula.split(':')[1].split()[0])
                            if 'High' in formula and item.get('Activity Level') == 'High':
                                multiplier = 1.20  # For Distributor Surge
                            elif 'level' in formula.lower():
                                multiplier = 1 + float(item.get('Activity Level', 0.10))
                        output += f"- {desc}: Multiplier {multiplier:.2f}\n"
                    else:
                        output += f"- {desc}: Qualitative impact\n"

            output += "\nDISTRIBUTOR ACTIVITY:\n"
            activity = context.get('distributor_activity', {})
            if activity:
                desc = activity.get('Event Description', 'Unknown')
                formula = activity.get('Impact Magnitude Formula', '')
                multiplier = 1.0
                if 'Multiplier' in formula:
                    multiplier = float(formula.split(':')[1].split()[0])
                    if 'High' in formula and activity.get('Activity Level') == 'High':
                        multiplier = 1.20
                output += f"- {desc}: Multiplier {multiplier:.2f}\n"
            else:
                output += "- No distributor activity\n"

            return output.strip()
        except Exception as e:
            return f"Error fetching context: {str(e)}"

class ForecastCalculatorTool(BaseTool):
    """Tool to help with forecast calculations"""
    name: str = "Forecast Calculator"
    description: str = "Applies multiple adjustments to a base forecast"

    def _run(self, base_forecast: float, multipliers: list) -> float:
        """Calculate adjusted forecast with multiple multipliers"""
        try:
            adjusted = base_forecast
            for m in multipliers:
                adjusted *= m
            return round(adjusted, 2)
        except Exception as e:
            return f"Error in calculation: {str(e)}"

global_mock_db = db  # Use 'db' from previous cell
context_tool = ContextFetcherTool()
calc_tool = ForecastCalculatorTool()

print("Testing Context Fetcher Tool:")
test_context = context_tool._run('AC-X1', 'North', 35)
print(test_context)

print("\nTesting Forecast Calculator Tool:")
test_calc = calc_tool._run(200, [1.10, 0.85])  # Example: promotion (+10%) and stockout (-15%)
print(f"Base: 200, Multipliers: [1.10, 0.85] = {test_calc}")

Testing Context Fetcher Tool:

Context for product=AC-X1, region=North, week=35:
  Promotions: 0, Stockouts: 0, Events: 1, Distributor activity: None
CONTEXT FOR AC-X1 in North - Week 35:

PROMOTIONS:

STOCKOUTS:

EVENTS:
- Increased purchases for electronics in Back to School campaign in North: Qualitative impact

DISTRIBUTOR ACTIVITY:
- No distributor activity

Testing Forecast Calculator Tool:
Base: 200, Multipliers: [1.10, 0.85] = 187.0


In [8]:
import ollama
from crewai import Agent, Task, Crew

class OllamaLLM:
    """Custom LLM wrapper for Ollama integration"""
    def __init__(self, model_name='llama3.2'):
        self.model_name = model_name
    
    def __call__(self, prompt: str) -> str:
        try:
            response = ollama.chat(
                model=self.model_name,
                messages=[{'role': 'user', 'content': prompt}]
            )
            return response['message']['content']
        except Exception as e:
            return f"LLM Error: {str(e)}"

class ForecastingPipeline:
    """Complete forecasting pipeline with agents, tasks, and crew"""
    def __init__(self, mock_db):
        self.mock_db = mock_db
        self.llm = OllamaLLM()
        self.setup_tools()
        self.setup_agents()
        self.setup_tasks()
        self.setup_crew()
    
    def setup_tools(self):
        """Initialize custom tools"""
        self.context_tool = ContextFetcherTool()  # Already initialized with global_mock_db
        self.calc_tool = ForecastCalculatorTool()
    
    def setup_agents(self):
        """Initialize the three required agents"""
        self.context_fetcher_agent = Agent(
            role='Business Context Analyst',
            goal='Fetch and analyze business context for accurate demand forecasting',
            backstory="""Expert in retail operations and supply chain, gathering promotions, stockouts, distributor activities, and events.""",
            verbose=True,
            allow_delegation=False,
            tools=[self.context_tool],
            llm=self.llm
        )
        self.forecast_enhancer_agent = Agent(
            role='Demand Forecasting Specialist',
            goal='Enhance baseline forecasts using business context multipliers',
            backstory="""15+ years in retail forecasting, adjusting demand based on business factors like promotions and stockouts.""",
            verbose=True,
            allow_delegation=False,
            tools=[self.calc_tool],
            llm=self.llm
        )
        self.explanation_agent = Agent(
            role='Business Intelligence Communicator',
            goal='Explain forecast adjustments clearly for stakeholders',
            backstory="""Expert in translating complex insights into actionable business recommendations.""",
            verbose=True,
            allow_delegation=False,
            llm=self.llm
        )
    
    def setup_tasks(self):
        """Define tasks for each agent"""
        self.context_task = Task(
            description="""Fetch business context for:
            Product: {product_code}
            Region: {region}
            Week: {week}
            Include promotions, stockouts, distributor activity, and events with their multipliers.""",
            agent=self.context_fetcher_agent,
            expected_output="Summary of business context with event descriptions and multipliers"
        )
        self.forecast_task = Task(
            description="""Enhance the baseline forecast using the business context:
            Baseline Forecast: {base_forecast} units
            Context: {context_summary}
            Extract multipliers from the context (e.g., 1.20 for promotions, 0.85 for stockouts).
            Use the forecast calculator tool to apply all multipliers.
            Provide:
            1. Adjusted forecast
            2. List of applied multipliers
            3. Reasoning for adjustments""",
            agent=self.forecast_enhancer_agent,
            expected_output="Enhanced forecast, multipliers, and reasoning",
            context=[self.context_task]
        )
        self.explanation_task = Task(
            description="""Explain the forecast adjustment:
            Original Forecast: {base_forecast}
            Enhanced Forecast: {enhanced_forecast}
            Context: {context_summary}
            Provide:
            1. Executive summary of the change
            2. Key business drivers (e.g., promotions, stockouts)
            3. Confidence level and risks
            4. Stakeholder recommendations""",
            agent=self.explanation_agent,
            expected_output="Business-friendly explanation with summary, drivers, risks, and recommendations",
            context=[self.context_task, self.forecast_task]
        )
    
    def setup_crew(self):
        """Initialize the crew with agents and tasks"""
        self.crew = Crew(
            agents=[self.context_fetcher_agent, self.forecast_enhancer_agent, self.explanation_agent],
            tasks=[self.context_task, self.forecast_task, self.explanation_task],
            verbose=True,
            process="sequential"
        )
    
    def run_forecast_enhancement(self, product_code: str, region: str, week: int, base_forecast: float):
        """Execute the forecasting pipeline"""
        inputs = {
            'product_code': product_code,
            'region': region,
            'week': week,
            'base_forecast': base_forecast
        }
        print(f"Enhancing forecast for {product_code} in {region}, Week {week}, Baseline: {base_forecast}")
        try:
            result = self.crew.kickoff(inputs)
            return {
                'success': True,
                'product_code': product_code,
                'region': region,
                'week': week,
                'base_forecast': base_forecast,
                'result': result
            }
        except Exception as e:
            return {
                'success': False,
                'error': str(e),
                'inputs': inputs
            }

print("Components ready: 3 Agents (Context, Forecast, Explanation)")

Components ready: 3 Agents (Context, Forecast, Explanation)


In [10]:
def create_forecast_tasks(product_code: str, region: str, week: int, base_forecast: float):
    """Create tasks for the forecast enhancement process"""
    
    # Task 1: Context Fetching
    context_task = Task(
        description=f"""
        Fetch comprehensive business context for:
        - Product: {product_code}
        - Region: {region}  
        - Week: {week}
        
        Use the Business Context Fetcher tool to gather information about:
        1. Active promotions and discount levels
        2. Any stockout situations
        3. Distributor activity and performance
        4. Seasonal events or market conditions
        
        Provide a detailed summary of all relevant business factors that could impact demand.
        """,
        agent=context_fetcher_agent,
        expected_output="Detailed business context report with all relevant factors affecting demand"
    )
    
    # Task 2: Forecast Enhancement
    enhancement_task = Task(
        description=f"""
        Based on the business context provided, enhance the baseline forecast of {base_forecast} units.
        
        Consider the following factors for adjustment:
        - Promotions typically increase demand by 15-40%
        - Stockouts in previous weeks can create increased demand (10-25% increase)
        - High distributor activity can boost sales by 10-20%
        - Seasonal events can have varying impacts (10-50% increase)
        - Multiple positive factors should be considered cumulatively but with diminishing returns
        
        Provide:
        1. Adjusted forecast number
        2. Percentage change from baseline
        3. Key factors driving the adjustment
        
        Use the Forecast Calculator tool if needed for precise calculations.
        """,
        agent=forecast_enhancer_agent,
        expected_output="Enhanced forecast with numerical value and key adjustment factors",
        context=[context_task]
    )
    
    # Task 3: Explanation Generation
    explanation_task = Task(
        description=f"""
        Create a comprehensive explanation for the forecast adjustment from {base_forecast} units.
        
        Your explanation should include:
        1. Executive Summary (2-3 sentences)
        2. Key Business Drivers (bullet points)
        3. Risk Factors to Monitor
        4. Recommended Actions for stakeholders
        5. Confidence Level (High/Medium/Low) with reasoning
        
        Make the explanation clear for business stakeholders who may not have technical forecasting background.
        Focus on actionable insights and business implications.
        """,
        agent=explanation_agent,
        expected_output="Clear, comprehensive business explanation with actionable insights",
        context=[context_task, enhancement_task]
    )
    
    return [context_task, enhancement_task, explanation_task]

print("Task creation function defined successfully!")
print("Tasks will be created for:")
print("   1. Context fetching and analysis")
print("   2. Forecast enhancement based on context")  
print("   3. Business explanation generation")

Task creation function defined successfully!
Tasks will be created for:
   1. Context fetching and analysis
   2. Forecast enhancement based on context
   3. Business explanation generation


In [20]:
from crewai.tools import BaseTool  # Make sure this is the correct import

# Define ForecastCalculatorTool
class ForecastCalculatorTool(BaseTool):
    name: str = "Forecast Calculator"
    description: str = "Calculates adjusted forecasts using context factors like promotions, stockouts, activity, and events"

    def _run(self, base_forecast: float, adjustments: list) -> dict:
        try:
            adjusted_forecast = base_forecast
            cumulative_factor = 1.0
            for adj in adjustments:
                cumulative_factor *= (1 + adj)
            adjusted_forecast *= cumulative_factor
            percentage_change = (adjusted_forecast - base_forecast) / base_forecast
            return {
                "adjusted_forecast": round(adjusted_forecast),
                "percentage_change": round(percentage_change * 100, 2)
            }
        except Exception as e:
            return {"error": str(e)}

# Define ContextFetcherTool
class ContextFetcherTool(BaseTool):
    name: str = "Business Context Fetcher"
    description: str = "Fetches business context including promotions, stockouts, distributor activity, and events for a given product, region, and week"
    
    def __init__(self, mock_db=None, **kwargs):
        super().__init__(**kwargs)
        if mock_db is not None:
            global global_mock_db
            global_mock_db = mock_db

    def _run(self, product_code: str, region: str, week: int) -> str:
        try:
            context = global_mock_db.get_context(product_code, region, week)
            formatted_context = f"""
BUSINESS CONTEXT FOR {product_code} in {region} - Week {week}:

PROMOTIONS:
"""
            if context['promotions']:
                for promo in context['promotions']:
                    formatted_context += f"- {promo.get('promotion_type', 'Unknown')}: {promo.get('discount_percent', 0)}% discount active\n"
            else:
                formatted_context += "- No active promotions\n"

            formatted_context += "\nSTOCKOUTS:\n"
            if context['stockouts']:
                for stockout in context['stockouts']:
                    days = stockout.get('days_out_of_stock', 'Unknown')
                    formatted_context += f"- Out of stock for {days} days\n"
            else:
                formatted_context += "- No stockouts reported\n"

            formatted_context += "\nDISTRIBUTOR ACTIVITY:\n"
            activity = context.get('distributor_activity', {})
            if activity:
                formatted_context += f"- Activity Level: {activity.get('activity_level', 'N/A')}\n"
                formatted_context += f"- New Distributors: {activity.get('new_distributors', 'N/A')}\n"
                performance = activity.get('distributor_performance')
                formatted_context += f"- Performance Score: {performance:.2f}\n" if performance else "- Performance Score: N/A\n"
            else:
                formatted_context += "- No distributor activity data\n"

            formatted_context += "\nSEASONAL EVENTS:\n"
            if context['events']:
                for event in context['events']:
                    formatted_context += f"- {event.get('event_name', 'Unknown Event')}: {event.get('impact_level', 'N/A')} impact expected\n"
            else:
                formatted_context += "- No seasonal events\n"

            return formatted_context.strip()
        except Exception as e:
            return f"Error fetching context: {str(e)}"


In [22]:
# Define ForecastCalculatorTool
class ForecastCalculatorTool(BaseTool):
    name: str = "Forecast Calculator"
    description: str = "Calculates adjusted forecasts using context factors like promotions, stockouts, activity, and events"

    def _run(self, base_forecast: float, adjustments: list) -> dict:
        try:
            adjusted_forecast = base_forecast
            cumulative_factor = 1.0
            for adj in adjustments:
                cumulative_factor *= (1 + adj)
            adjusted_forecast *= cumulative_factor
            percentage_change = (adjusted_forecast - base_forecast) / base_forecast
            return {
                "adjusted_forecast": round(adjusted_forecast),
                "percentage_change": round(percentage_change * 100, 2)
            }
        except Exception as e:
            return {"error": str(e)}

# Define ContextFetcherTool
class ContextFetcherTool(BaseTool):
    name: str = "Business Context Fetcher"
    description: str = "Fetches business context including promotions, stockouts, distributor activity, and events for a given product, region, and week"
    
    def __init__(self, mock_db=None, **kwargs):
        super().__init__(**kwargs)
        if mock_db is not None:
            global global_mock_db
            global_mock_db = mock_db

    def _run(self, product_code: str, region: str, week: int) -> str:
        try:
            context = global_mock_db.get_context(product_code, region, week)
            formatted_context = f"""
BUSINESS CONTEXT FOR {product_code} in {region} - Week {week}:

PROMOTIONS:
"""
            if context['promotions']:
                for promo in context['promotions']:
                    formatted_context += f"- {promo.get('promotion_type', 'Unknown')}: {promo.get('discount_percent', 0)}% discount active\n"
            else:
                formatted_context += "- No active promotions\n"

            formatted_context += "\nSTOCKOUTS:\n"
            if context['stockouts']:
                for stockout in context['stockouts']:
                    days = stockout.get('days_out_of_stock', 'Unknown')
                    formatted_context += f"- Out of stock for {days} days\n"
            else:
                formatted_context += "- No stockouts reported\n"

            formatted_context += "\nDISTRIBUTOR ACTIVITY:\n"
            activity = context.get('distributor_activity', {})
            if activity:
                formatted_context += f"- Activity Level: {activity.get('activity_level', 'N/A')}\n"
                formatted_context += f"- New Distributors: {activity.get('new_distributors', 'N/A')}\n"
                performance = activity.get('distributor_performance')
                formatted_context += f"- Performance Score: {performance:.2f}\n" if performance else "- Performance Score: N/A\n"
            else:
                formatted_context += "- No distributor activity data\n"

            formatted_context += "\nSEASONAL EVENTS:\n"
            if context['events']:
                for event in context['events']:
                    formatted_context += f"- {event.get('event_name', 'Unknown Event')}: {event.get('impact_level', 'N/A')} impact expected\n"
            else:
                formatted_context += "- No seasonal events\n"

            return formatted_context.strip()
        except Exception as e:
            return f"Error fetching context: {str(e)}"

# Define ForecastEnhancementCrew
class ForecastEnhancementCrew:
    """Simple agent crew to run forecasting steps"""
    def __init__(self, agents):
        self.agents = agents

    def run(self, input_data):
        result = input_data
        for agent in self.agents:
            result = agent.run(result)
        return result

pipeline = ForecastingPipeline(mock_db)

forecast_crew = ForecastEnhancementCrew([
    pipeline.context_fetcher_agent,
    pipeline.forecast_enhancer_agent,
    pipeline.explanation_agent
])

print("Forecast Enhancement Crew initialized successfully!")


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Forecast Enhancement Crew initialized successfully!


In [24]:
context_tool = ContextFetcherTool(mock_db=mock_db)
calc_tool = ForecastCalculatorTool()
llm = OllamaLLM()

def quick_test():
    """Quick test without full crew execution"""
    
    print("QUICK FUNCTIONALITY TEST")
    print("=" * 40)

    print("1. Testing Mock Database...")
    context = mock_db.get_context('AC-X1', 'North', 35)
    print(f"   Context retrieved: {len(context)} fields")
    
    print("2. Testing Context Tool...")
    formatted_context = context_tool._run('AC-X1', 'North', 35)
    print(f"   Formatted context length: {len(formatted_context)} characters")
    
    print("3. Testing Calculator Tool...")
    # Adjustments as a list of floats (e.g. 25% = 0.25)
    calc_result = calc_tool._run(200, [0.25])
    print(f"   Calculation result: {calc_result}")
    
    print("4. Testing LLM Connection...")
    try:
        test_response = llm("What is 2+2?")
        print(f"   LLM Response: {test_response[:50]}...")
    except Exception as e:
        print(f"   LLM Error: {str(e)}")
    
    print("\nQuick test completed!")

def run_forecast_enhancement_demo():
    """Demo function to test the complete system"""

    print("FORECAST ENHANCEMENT DEMO")
    print("=" * 60)
    
    test_scenarios = [
        {
            'product_code': 'AC-X1',
            'region': 'North',
            'week': 35,  # Back to school season
            'base_forecast': 200,
            'description': 'Air Conditioner during Back to School season'
        },
        {
            'product_code': 'TV-S2',
            'region': 'South',
            'week': 45,
            'base_forecast': 150,
            'description': 'Smart TV during Diwali season'
        },
        {
            'product_code': 'REF-C3',
            'region': 'West',
            'week': 20,
            'base_forecast': 100,
            'description': 'Refrigerator during Summer Sale'
        }
    ]
    
    results = []
    
    for i, scenario in enumerate(test_scenarios, 1):
        print(f"\nTEST SCENARIO {i}: {scenario['description']}")
        print("-" * 40)
        
        context = mock_db.get_context(
            scenario['product_code'], 
            scenario['region'], 
            scenario['week']
        )
        
        print("Business Context Preview:")
        if context['promotions']:
            print(f"   • Promotions: {len(context['promotions'])} active")
        if context['stockouts']:
            print(f"   • Stockouts: {len(context['stockouts'])} reported")
        if context['events']:
            print(f"   • Events: {context['events'][0]['event_name'] if context['events'] else 'None'}")
        if context['distributor_activity']:
            print(f"   • Distributor Activity: {context['distributor_activity']['activity_level']}")
        
        try:
            result = forecast_crew.enhance_forecast(
                product_code=scenario['product_code'],
                region=scenario['region'],
                week=scenario['week'],
                base_forecast=scenario['base_forecast']
            )
            
            if result:
                results.append({**scenario, **result})
                
                print("\nRESULTS SUMMARY:")
                print(f"   Base Forecast: {result['base_forecast']} units")
                print(f"   Enhanced Forecast: {result['enhanced_forecast']} units")
                print(f"   Adjustment: {result['adjustment_percentage']:.1f}%")
                print("\nExplanation:")
                print(f"   {result['explanation'][:200]}...")
            else:
                print("Failed to generate enhanced forecast")
                
        except Exception as e:
            print(f"Error in scenario {i}: {str(e)}")
        
        print("\n" + "="*60)
    
    return results

print("Choose test type:")
print("1. Quick Test (basic functionality)")
print("2. Full Demo (complete crew execution)")

quick_test()

# Uncomment below to run full demo after you have forecast_crew defined properly
# run_forecast_enhancement_demo()


Choose test type:
1. Quick Test (basic functionality)
2. Full Demo (complete crew execution)
QUICK FUNCTIONALITY TEST
1. Testing Mock Database...

Context for product=AC-X1, region=North, week=35:
  Promotions: 0, Stockouts: 0, Events: 1, Distributor activity: None
   Context retrieved: 7 fields
2. Testing Context Tool...

Context for product=AC-X1, region=North, week=35:
  Promotions: 0, Stockouts: 0, Events: 1, Distributor activity: None
   Formatted context length: 227 characters
3. Testing Calculator Tool...
   Calculation result: {'adjusted_forecast': 250, 'percentage_change': 25.0}
4. Testing LLM Connection...
   LLM Response: 2 + 2 = 4...

Quick test completed!


In [42]:
class ForecastEnhancementInterface:
    def __init__(self, test_mode=False):
    
        if test_mode:
            self.db = self._make_test_db()
        else:
            try:
                self.db = mock_db
            except NameError:
                raise RuntimeError("mock_db is not defined")
        

        class MockCrew:
            def enhance_forecast(self, p_code, region, week, base):
                return {
                    'base_forecast': base,
                    'enhanced_forecast': base + 50,
                    'adjustment_percentage': 25.0,
                    'explanation': "Sample AI-driven boost based on promotions and seasonal trends."
                }
        self.crew = MockCrew()

    def _make_test_db(self):
        """Returns a fake DB with sample products, regions, and get_context() data."""
        class TestDB:
            products = ['AC-X1', 'TV-S2', 'RF-M3']
            regions = ['North', 'South', 'East']
            def get_context(self, p_code, region, week):
            
                return {
                    'promotions': [
                        {'promotion_type': 'Summer Sale', 'discount_percent': 30}
                    ],
                    'stockouts': [
                        {'days_out_of_stock': 6}
                    ],
                    'distributor_activity': {
                        'activity_level': 'High',
                        'new_distributors': 2,
                        'distributor_performance': 0.85
                    },
                    'events': [
                        {'event_name': 'Back to School', 'impact_level': 'High'}
                    ]
                }
        return TestDB()



    def manual_forecast_adjustment(self, p_code, region, week, base):
    
        ctx = self.db.get_context(p_code, region, week)
        factor = 1.0
        reasons = []

        for promo in ctx.get('promotions', []):
            d = float(promo.get('discount_percent', 0))
            if d >= 30:
                factor *= 1.4; reasons.append(f"High discount ({d}%): +40%")
            elif d >= 20:
                factor *= 1.25; reasons.append(f"Medium discount ({d}%): +25%")
            else:
                factor *= 1.15; reasons.append(f"Low discount ({d}%): +15%")

        for s in ctx.get('stockouts', []):
            days = int(s.get('days_out_of_stock', 0))
            if days >= 5:
                factor *= 1.3; reasons.append(f"Long stockout ({days} days): +30%")
            else:
                factor *= 1.15; reasons.append(f"Short stockout ({days} days): +15%")

        da = ctx.get('distributor_activity', {})
        lvl = da.get('activity_level', '').lower()
        if lvl == 'high':
            factor *= 1.2; reasons.append("High distributor activity: +20%")
        elif lvl:
            factor *= 1.1; reasons.append(f"{da.get('activity_level')}: +10%")

        for e in ctx.get('events', []):
            lvl = str(e.get('impact_level', '')).lower()
            name = e.get('event_name', 'Event')
            if lvl == 'high':
                factor *= 1.3; reasons.append(f"{name}: +30%")
            else:
                factor *= 1.15; reasons.append(f"{name}: +15%")

        if factor > 1.5:
            factor = 1.0 + (factor - 1.0) * 0.8
            reasons.append("Applied diminishing returns")

        enhanced = round(base * factor)
        adj_pct = round((factor - 1.0) * 100, 1)
        return enhanced, adj_pct, reasons


# Example usage in test mode:
interface = ForecastEnhancementInterface(test_mode=True)
print("Interactive Interface Ready in TEST MODE!\n")
# Directly call manual adjustment for a quick check:
e, a, r = interface.manual_forecast_adjustment('AC-X1', 'North', 35, 200)
print(f"Sample Manual Enhancement:\n  Base: 200  Enhanced: {e}  Adjustment: {a}%\n  Reasons: {r}")


Interactive Interface Ready in TEST MODE!

Sample Manual Enhancement:
  Base: 200  Enhanced: 494  Adjustment: 147.1%
  Reasons: ['High discount (30.0%): +40%', 'Long stockout (6 days): +30%', 'High distributor activity: +20%', 'Back to School: +30%', 'Applied diminishing returns']


In [50]:
original_get_context = mock_db.get_context  # backup original method

def mock_get_context(product, region, week):
    if (product, region, week) == ('TV-S2', 'North', 45):
        return {
            'promotions': [{'type': 'Diwali Sale', 'intensity': 'high', 'discount_percent': 35}],
            'events': [{'event_name': 'Diwali', 'impact_level': 'High'}],
            'distributor_activity': {'activity_level': 'very_high'}
        }
    elif (product, region, week) == ('REF-C3', 'East', 15):
        return {
            'promotions': [{'type': 'Regular Promo', 'intensity': 'low', 'discount_percent': 10}],
            'events': [],
            'distributor_activity': {'activity_level': 'normal'}
        }
    else:
        return {}


mock_db.get_context = mock_get_context

def demonstrate_complete_workflow():
    """Demonstrate the complete forecast enhancement workflow"""
    
    print(" COMPLETE WORKFLOW DEMONSTRATION")
    print("=" * 60)
    
    print("\n EXAMPLE 1: High-Impact Scenario")
    print("-" * 40)
    
    product = 'TV-S2'
    region = 'North'
    week = 45  
    base_forecast = 150
    
    print(f"Scenario: {product} in {region}, Week {week} (Diwali season)")
    print(f"Base Forecast: {base_forecast} units")
     
    context = mock_db.get_context(product, region, week)
    if context is None:
        context = {}
    promotions = context.get('promotions', [])
    events = context.get('events', [])
    distributor_activity = context.get('distributor_activity', None)
    
    print(f"\nBusiness Context:")
    print(f"• Promotions: {len(promotions)} active")
    print(f"• Events: {[e.get('event_name', 'N/A') for e in events]}")
    print(f"• Distributor Activity: {distributor_activity.get('activity_level', 'N/A') if distributor_activity else 'N/A'}")
    
    enhanced, adj_pct, reasons = interface.manual_forecast_adjustment(product, region, week, base_forecast)
    print(f"\nEnhancement Results:")
    print(f"• Enhanced Forecast: {enhanced} units")
    print(f"• Adjustment: {adj_pct:+.1f}%")
    print(f"• Key Factors: {len(reasons)} considered")
    if reasons:
        print("  - Reasons:")
        for r in reasons:
            print(f"    * {r}")
    
    print("\n\n EXAMPLE 2: Low-Impact Scenario")
    print("-" * 40)
    
    product = 'REF-C3'
    region = 'East'
    week = 15  
    base_forecast = 80
    
    print(f"Scenario: {product} in {region}, Week {week} (regular week)")
    print(f"Base Forecast: {base_forecast} units")
    
    context = mock_db.get_context(product, region, week)
    if context is None:
        context = {}
    promotions = context.get('promotions', [])
    events = context.get('events', [])
    distributor_activity = context.get('distributor_activity', None)
    
    enhanced, adj_pct, reasons = interface.manual_forecast_adjustment(product, region, week, base_forecast)
    
    print(f"\nEnhancement Results:")
    print(f"• Enhanced Forecast: {enhanced} units")
    print(f"• Adjustment: {adj_pct:+.1f}%")
    print(f"• Impact: {'Low' if abs(adj_pct) < 15 else 'High'}")
    if reasons:
        print("  - Reasons:")
        for r in reasons:
            print(f"    * {r}")

def show_api_usage_examples():
    """Show different ways to use the forecast enhancement system"""
    
    print("\n API USAGE EXAMPLES")
    print("=" * 50)
    
    print("\n1️ SINGLE FORECAST ENHANCEMENT:")
    print("# Manual rule-based enhancement")
    print("enhanced, adj_pct, reasons = interface.manual_forecast_adjustment(")
    print("    product_code='AC-X1',")
    print("    region='North',")
    print("    week=35,")
    print("    base_forecast=200")
    print("print(f'Enhanced: {enhanced}, Adjustment: {adj_pct}%')")
    
    print("\n AI AGENT ENHANCEMENT:")
    print("# Using CrewAI agents (requires Ollama)")
    print("result = forecast_crew.enhance_forecast(")
    print("    product_code='TV-S2',")
    print("    region='South',")
    print("    week=45,")
    print("    base_forecast=150")
    print("print(result)")
    
    print("\n INTERACTIVE SESSION:")
    print("# Interactive command-line interface")
    print("interface.run_interactive_session()")

def create_batch_forecast_processor():
    """Create a batch processor for multiple forecasts"""
    
    class BatchForecastProcessor:
        def __init__(self):
            self.interface = ForecastEnhancementInterface()
            self.results = []
        
        def process_batch(self, forecast_requests):
            """Process a batch of forecast requests"""
            print(f" Processing {len(forecast_requests)} forecasts...")
            
            for i, request in enumerate(forecast_requests, 1):
                print(f"\nProcessing {i}/{len(forecast_requests)}: {request['product']}")
                
                enhanced, adj_pct, reasons = self.interface.manual_forecast_adjustment(
                    request['product'],
                    request['region'],
                    request['week'],
                    request['base_forecast']
                )
                
                result = {
                    **request,
                    'enhanced_forecast': enhanced,
                    'adjustment_percentage': adj_pct,
                    'adjustment_reasons': reasons,
                    'confidence': 'High' if len(reasons) >= 2 else 'Medium'
                }
                
                self.results.append(result)
            
            return self.results
        
        def export_results(self):
            """Export results to DataFrame"""
            import pandas as pd
            return pd.DataFrame(self.results)
    
    return BatchForecastProcessor()

def create_summary_report():
    """Create a summary of the implemented system"""
    
    print("\n SYSTEM SUMMARY REPORT")
    print("=" * 60)
    
    print("\n ARCHITECTURE COMPONENTS:")
    print("• Mock Frappe Database - Simulates business context storage")
    print("• Context Fetcher Tool - Retrieves relevant business data")
    
    print("\n AI AGENTS:")
    print("1. Context Fetcher Agent - Analyzes business environment")
    print("2. Forecast Enhancement Agent - Adjusts forecasts intelligently")  
    print("3. Explanation Generator Agent - Provides business insights")
    
    print("\n BUSINESS FACTORS CONSIDERED:")
    print("• Promotional Activities (10-40% impact)")
    print("• Stock-out Situations (10-30% pent-up demand)")
    print("• Distributor Performance (10-20% impact)")
    print("• Seasonal Events (10-50% impact)")
    print("• Diminishing Returns Logic (prevents over-adjustment)")


print(" RUNNING DEMONSTRATIONS...\n")

demonstrate_complete_workflow()

show_api_usage_examples()

batch_processor = create_batch_forecast_processor()
print(f"\nBatch processor created: {type(batch_processor).__name__}")

create_summary_report()

# Restore the original get_context method if needed
# mock_db.get_context = original_get_context


 RUNNING DEMONSTRATIONS...

 COMPLETE WORKFLOW DEMONSTRATION

 EXAMPLE 1: High-Impact Scenario
----------------------------------------
Scenario: TV-S2 in North, Week 45 (Diwali season)
Base Forecast: 150 units

Business Context:
• Promotions: 1 active
• Events: ['Diwali']
• Distributor Activity: very_high

Enhancement Results:
• Enhanced Forecast: 371 units
• Adjustment: +147.1%
• Key Factors: 5 considered
  - Reasons:
    * High discount (30.0%): +40%
    * Long stockout (6 days): +30%
    * High distributor activity: +20%
    * Back to School: +30%
    * Applied diminishing returns


 EXAMPLE 2: Low-Impact Scenario
----------------------------------------
Scenario: REF-C3 in East, Week 15 (regular week)
Base Forecast: 80 units

Enhancement Results:
• Enhanced Forecast: 198 units
• Adjustment: +147.1%
• Impact: High
  - Reasons:
    * High discount (30.0%): +40%
    * Long stockout (6 days): +30%
    * High distributor activity: +20%
    * Back to School: +30%
    * Applied diminishi

In [30]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import math

def evaluate_forecasts(forecast_requests, actuals, interface):
    """
    Evaluate baseline vs enhanced forecasts using MAPE and RMSE.
    Uses your ForecastEnhancementInterface.manual_adjustment() for enhancement.
    
    Args:
        forecast_requests: list of dicts with keys: product, region, week, base_forecast
        actuals: list of actual sales numbers aligned with forecast_requests
        interface: instance of ForecastEnhancementInterface
    
    Returns:
        List of dicts with evaluation results per forecast.
    """
    results = []
    for i, req in enumerate(forecast_requests):
        p = req['product']
        r = req['region']
        w = req['week']
        base = req['base_forecast']
        actual = actuals[i]

        # Get enhanced forecast and adjustment info
        enhanced, adj_pct, reasons = interface.manual_forecast_adjustment(p, r, w, base)

        mape_base = mean_absolute_percentage_error([actual], [base]) * 100
        mape_enh = mean_absolute_percentage_error([actual], [enhanced]) * 100

        rmse_base = math.sqrt(mean_squared_error([actual], [base]))
        rmse_enh = math.sqrt(mean_squared_error([actual], [enhanced]))

        entry = {
            'product': p,
            'region': r,
            'week': w,
            'base_forecast': base,
            'enhanced_forecast': enhanced,
            'actual': actual,
            'MAPE_base (%)': mape_base,
            'MAPE_enhanced (%)': mape_enh,
            'RMSE_base': rmse_base,
            'RMSE_enhanced': rmse_enh,
            'adjustment_pct': adj_pct,
            'adjustment_reasons': reasons
        }
        results.append(entry)

    return results


def print_evaluation_results(results):
    for res in results:
        print(f"Product: {res['product']}, Region: {res['region']}, Week: {res['week']}")
        print(f"Base Forecast: {res['base_forecast']} units")
        print(f"Enhanced Forecast: {res['enhanced_forecast']} units")
        print(f"Actual Sales: {res['actual']} units")
        print(f"Adjustment: {res['adjustment_pct']:.1f}%")
        print("Adjustment Reasons:")
        for reason in res['adjustment_reasons']:
            print(f"  • {reason}")

        print(f"MAPE (Base): {res['MAPE_base (%)']:.2f}%")
        print(f"MAPE (Enhanced): {res['MAPE_enhanced (%)']:.2f}%")
        print(f"RMSE (Base): {res['RMSE_base']:.2f}")
        print(f"RMSE (Enhanced): {res['RMSE_enhanced']:.2f}")

        if res['MAPE_enhanced (%)'] > res['MAPE_base (%)']:
            print("Warning: Enhanced forecast is less accurate than base forecast!")

        print("-" * 50)


# Example usage (make sure you have your interface instance defined):
forecast_requests = [
    {'product': 'AC-X1', 'region': 'North', 'week': 35, 'base_forecast': 200},
    {'product': 'TV-S2', 'region': 'South', 'week': 40, 'base_forecast': 150},
]
actuals = [250, 140]

evaluation_results = evaluate_forecasts(forecast_requests, actuals, interface)
print_evaluation_results(evaluation_results)


Product: AC-X1, Region: North, Week: 35
Base Forecast: 200 units
Enhanced Forecast: 200 units
Actual Sales: 250 units
Adjustment: 0.0%
Adjustment Reasons:
MAPE (Base): 20.00%
MAPE (Enhanced): 20.00%
RMSE (Base): 50.00
RMSE (Enhanced): 50.00
--------------------------------------------------
Product: TV-S2, Region: South, Week: 40
Base Forecast: 150 units
Enhanced Forecast: 150 units
Actual Sales: 140 units
Adjustment: 0.0%
Adjustment Reasons:
MAPE (Base): 7.14%
MAPE (Enhanced): 7.14%
RMSE (Base): 10.00
RMSE (Enhanced): 10.00
--------------------------------------------------
